# Step 0 — Data Preparation & Alignment

**Purpose:** Run iggnition alignment on all raw paired sequences, define naive/memory subsets,
compute per-sequence mutation summary statistics, apply QC filters, and save the master
analysis table used by all downstream steps.

**Output:** `processed/aligned_master.parquet`

**Design reference:** DESIGN.md §Step 0

In [18]:
import polars as pl
import numpy as np
import iggnition
from pathlib import Path
import time
from tqdm.auto import tqdm

In [3]:
DATA_DIR     = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RAW_FILE     = DATA_DIR / "20260309_complete_database_with_lineages_and_subsets.parquet"
PROC_DIR     = DATA_DIR / "processed"
PROC_DIR.mkdir(exist_ok=True)

# Intermediate iggnition outputs (expensive to recompute)
MUTATED_FILE  = PROC_DIR / "iggnition_sequences.parquet"
GERMLINE_FILE = PROC_DIR / "iggnition_germlines.parquet"
MASTER_FILE   = PROC_DIR / "aligned_master.parquet"

## 0.1  Load raw data

In [4]:
df = pl.read_parquet(RAW_FILE)
print(f"Total sequences: {df.height:,}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col!r:45s}  {str(df[col].dtype):<15}  "
          f"nulls={df[col].null_count():,}  "
          f"example={df[col][0]}")

Total sequences: 4,840,405

Columns (307):
  'name'                                         String           nulls=0  example=ACGATGTTCCCTAACC_d01_w83
  'sequence_id:0'                                String           nulls=0  example=ACGATGTTCCCTAACC_contig-0_d01_w83
  'v_gene:0'                                     String           nulls=0  example=IGHV3-9
  'd_gene:0'                                     String           nulls=555,540  example=IGHD6-13
  'j_gene:0'                                     String           nulls=0  example=IGHJ1
  'c_gene:0'                                     String           nulls=354,006  example=IGHM
  'cdr3_length:0'                                Int64            nulls=0  example=15
  'junction_aa:0'                                String           nulls=0  example=CAKDPAAAGPIQYFQHW
  'v_identity:0'                                 Float64          nulls=0  example=0.9730639730639731
  'v_identity_aa:0'                              Float64          nulls

## 0.2  Naive / memory subset definitions

Three operationally distinct definitions of naïve B cells:

| Label | Definition |
|---|---|
| `naive_bio` | Biological label: `subset == 'naive'` (CD27⁻ MACS separation) |
| `naive_comp` | Computational: IgM + V-gene identity ≥ 99% |
| `naive_strict` | Intersection of both (most conservative) |

All three are retained in the master table for comparative analyses.

In [5]:
df = df.with_columns([
    # Method 1: biological label from wetlab CD27- MACS separation
    (pl.col('subset') == 'naive').alias('naive_bio'),

    # Method 2: computational — IgM + < 1% SHM in V gene of heavy chain
    ((pl.col('c_gene:0') == 'IGHM') & (pl.col('v_identity:0') >= 0.99)).alias('naive_comp'),
])

df = df.with_columns([
    # Method 3: intersection of both
    (pl.col('naive_bio') & pl.col('naive_comp')).alias('naive_strict'),
])

N = df.height
for label in ('naive_bio', 'naive_comp', 'naive_strict'):
    n = df.filter(pl.col(label)).height
    print(f"{label:20s}: {n:>9,}  ({100*n/N:.1f}%)")

# Concordance: how often do bio and comp agree?
agree = df.filter(pl.col('naive_bio') == pl.col('naive_comp')).height
print(f"\nBio == Comp concordance: {agree:,} / {N:,} ({100*agree/N:.2f}%)")

bio_not_comp = df.filter(pl.col('naive_bio') & ~pl.col('naive_comp')).height
comp_not_bio = df.filter(pl.col('naive_comp') & ~pl.col('naive_bio')).height
print(f"Bio-naïve but NOT comp-naïve: {bio_not_comp:,}  (IgM-switched or mutated CD27- cells)")
print(f"Comp-naïve but NOT bio-naïve: {comp_not_bio:,}  (unmutated IgM in memory fraction)")

naive_bio           : 2,914,772  (60.2%)
naive_comp          : 2,141,846  (44.2%)
naive_strict        : 1,979,995  (40.9%)

Bio == Comp concordance: 3,735,806 / 4,840,405 (77.18%)
Bio-naïve but NOT comp-naïve: 692,383  (IgM-switched or mutated CD27- cells)
Comp-naïve but NOT bio-naïve: 161,851  (unmutated IgM in memory fraction)


## 0.3  iggnition alignment

Run iggnition on **all** sequences (naïve + memory) to produce IMGT/Aho-numbered nucleotide
alignments in wide format.  
Results are cached to parquet — skip if already computed.

In [6]:
if MUTATED_FILE.exists():
    print(f"Loading cached sequences from {MUTATED_FILE}")
    mutated = pl.read_parquet(MUTATED_FILE)
else:
    print("Running iggnition on sequences...")
    t0 = time.time()
    mutated, _ = iggnition.run(df, name_col="name", wide=True, verbose=True, human_readable=False)
    print(f"Done in {time.time()-t0:.0f}s — shape: {mutated.shape}")
    mutated.write_parquet(MUTATED_FILE)
    print(f"Saved → {MUTATED_FILE}")

mutated.head(3)

Running iggnition on sequences...


numbering:   0%|          | 0/9680810 [00:00<?, ? seq/s]

Done in 162s — shape: (4840454, 892)
Saved → /home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/processed/iggnition_sequences.parquet


seq_name,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12,H13,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26,H27,H28,H29,H30,H31,H32,H33,H34,H35,H36,…,L408,L409,L410,L411,L412,L413,L414,L415,L416,L417,L418,L419,L420,L421,L422,L423,L424,L425,L426,L427,L428,L429,L430,L431,L432,L433,L434,L435,L436,L437,L438,L439,L440,L441,L442,L443,L444
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""ACGATGTTCCCTAACC_d01_w83""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""CACAGTACATGAACCT_d05_w49""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,65,67,67,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""CCTAGCTTCGGCGCAT_d04_w2""",67,65,71,71,84,71,67,65,71,67,84,71,67,65,71,71,65,71,84,67,71,45,45,45,71,71,67,67,67,65,71,71,65,67,84,71,…,45,45,45,45,45,45,45,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45


In [7]:
if GERMLINE_FILE.exists():
    print(f"Loading cached germlines from {GERMLINE_FILE}")
    germline = pl.read_parquet(GERMLINE_FILE)
else:
    print("Running iggnition on germlines...")
    t0 = time.time()
    germline, _ = iggnition.run(
        df, paired=True,
        nt_col_heavy="germline:0",  aa_col_heavy="germline_aa:0",
        nt_col_light="germline:1",  aa_col_light="germline_aa:1",
        name_col="name", wide=True, verbose=True, human_readable=False,
    )
    print(f"Done in {time.time()-t0:.0f}s — shape: {germline.shape}")
    germline.write_parquet(GERMLINE_FILE)
    print(f"Saved → {GERMLINE_FILE}")

germline.head(3)

Running iggnition on germlines...


numbering:   0%|          | 0/9680810 [00:00<?, ? seq/s]

Done in 157s — shape: (4839991, 892)
Saved → /home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/processed/iggnition_germlines.parquet


seq_name,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12,H13,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26,H27,H28,H29,H30,H31,H32,H33,H34,H35,H36,…,L408,L409,L410,L411,L412,L413,L414,L415,L416,L417,L418,L419,L420,L421,L422,L423,L424,L425,L426,L427,L428,L429,L430,L431,L432,L433,L434,L435,L436,L437,L438,L439,L440,L441,L442,L443,L444
str,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
"""ACGATGTTCCCTAACC_d01_w83""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""CACAGTACATGAACCT_d05_w49""",71,65,65,71,84,71,67,65,71,67,84,71,71,84,71,71,65,71,84,67,84,45,45,45,71,71,71,71,71,65,71,71,67,84,84,71,…,45,45,45,45,45,45,45,65,67,67,71,84,67,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
"""CCTAGCTTCGGCGCAT_d04_w2""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,45,45,45,45,45,45,45,67,84,65,71,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45


## 0.4  Mutation matrix

Per-sequence, per-nucleotide-position mutation indicator: `sequence_nt − germline_nt`.
- `0`   = no change
- non-zero = nucleotide mutation (value encodes direction via ASCII difference)
- `null`   = position absent in alignment

In [ ]:
KEY = "seq_name"
pos_cols = [c for c in mutated.columns if c != KEY]

# Deduplicate iggnition outputs (49 sequences had germline numbering failures
# that caused extra rows in the sequence table; dedup before joining).
mutated_dedup  = mutated.unique(subset=[KEY], keep='first')
germline_dedup = germline.unique(subset=[KEY], keep='first')
print(f"After dedup: sequences={mutated_dedup.height:,}, germlines={germline_dedup.height:,}")

# Single inner join: sequence nt columns + germline nt columns (with _germ suffix)
# in one DataFrame. Row order is fixed by sort(KEY) so all downstream numpy arrays
# extracted from this table are guaranteed to be row-aligned.
aligned_joint = (
    mutated_dedup
    .join(germline_dedup, on=KEY, how="inner", suffix="_germ")
    .sort(KEY)
)
print(f"Aligned pairs: {aligned_joint.height:,} "
      f"(dropped {mutated_dedup.height - aligned_joint.height:,} without germline match)")

# Mutation matrix: sequence_nt − germline_nt per position.
# Must apply the difference HERE on aligned_joint; simply selecting pos_cols from
# aligned_joint would give raw sequence nucleotides (non-zero = "mutated" incorrectly).
mutations_sorted = (
    aligned_joint
    .with_columns([
        (pl.col(c) - pl.col(f"{c}_germ")).alias(c)
        for c in pos_cols
    ])
    .select([KEY] + pos_cols)
)
print(f"Mutation matrix: {mutations_sorted.shape}")

## 0.5  Region maps (Aho nucleotide coordinates)

Aho amino acid boundaries (from iggnition `regions.py`, branch `map`):

| Region | Heavy | Light |
|--------|-------|-------|
| FR1    | 1–25  | 1–25  |
| CDR1   | 26–38 | 26–38 |
| FR2    | 39–49 | 39–49 |
| CDR2   | 50–64 | 50–66 |
| FR3    | 65–108| 67–108|
| CDR3   | 109–137| 109–138|
| FR4    | 138–149| 139–148|

Nucleotide column formula: Aho AA position `k` → nt columns `{chain}{(k-1)*3+1}` … `{chain}{k*3}`

In [20]:
def aho_to_nt_cols(chain: str, aho_start: int, aho_end: int) -> list[str]:
    """Return nt column names for Aho AA positions aho_start..aho_end (inclusive)."""
    return [f"{chain}{i}" for i in range((aho_start - 1) * 3 + 1, aho_end * 3 + 1)]


# Heavy chain region columns (nucleotide-level)
H_REGIONS = {
    'FR1' : aho_to_nt_cols('H',   1,  25),
    'CDR1': aho_to_nt_cols('H',  26,  38),
    'FR2' : aho_to_nt_cols('H',  39,  49),
    'CDR2': aho_to_nt_cols('H',  50,  64),
    'FR3' : aho_to_nt_cols('H',  65, 108),
    'CDR3': aho_to_nt_cols('H', 109, 137),
    'FR4' : aho_to_nt_cols('H', 138, 149),
}

# Light chain region columns (nucleotide-level; K and L share Aho boundaries)
L_REGIONS = {
    'FR1' : aho_to_nt_cols('L',   1,  25),
    'CDR1': aho_to_nt_cols('L',  26,  38),
    'FR2' : aho_to_nt_cols('L',  39,  49),
    'CDR2': aho_to_nt_cols('L',  50,  66),
    'FR3' : aho_to_nt_cols('L',  67, 108),
    'CDR3': aho_to_nt_cols('L', 109, 138),
    'FR4' : aho_to_nt_cols('L', 139, 148),
}

# Convenience groupings
H_CDR_COLS = H_REGIONS['CDR1'] + H_REGIONS['CDR2'] + H_REGIONS['CDR3']
H_FWR_COLS = H_REGIONS['FR1']  + H_REGIONS['FR2']  + H_REGIONS['FR3'] + H_REGIONS['FR4']
L_CDR_COLS = L_REGIONS['CDR1'] + L_REGIONS['CDR2'] + L_REGIONS['CDR3']
L_FWR_COLS = L_REGIONS['FR1']  + L_REGIONS['FR2']  + L_REGIONS['FR3'] + L_REGIONS['FR4']

# Filter to columns actually present in the mutation matrix
present = set(mutations_sorted.columns)
H_CDR_COLS = [c for c in H_CDR_COLS if c in present]
H_FWR_COLS = [c for c in H_FWR_COLS if c in present]
L_CDR_COLS = [c for c in L_CDR_COLS if c in present]
L_FWR_COLS = [c for c in L_FWR_COLS if c in present]

print(f"H CDR cols: {len(H_CDR_COLS)} nt  ({len(H_CDR_COLS)//3} codons)")
print(f"H FWR cols: {len(H_FWR_COLS)} nt  ({len(H_FWR_COLS)//3} codons)")
print(f"L CDR cols: {len(L_CDR_COLS)} nt  ({len(L_CDR_COLS)//3} codons)")
print(f"L FWR cols: {len(L_FWR_COLS)} nt  ({len(L_FWR_COLS)//3} codons)")

H CDR cols: 171 nt  (57 codons)
H FWR cols: 276 nt  (92 codons)
L CDR cols: 180 nt  (60 codons)
L FWR cols: 264 nt  (88 codons)


## 0.6  Mutation summary counts

### 0.6a  Codon-level mutation counts by chain and region

A codon is counted as mutated if **any** of its 3 nucleotide positions differs from germline.
This gives counts in amino acid units (comparable across sequences regardless of CDR3 length).

In [21]:
GAP = 45  # iggnition encodes alignment gaps as ASCII 45 ('-'), not null

def codon_mut_count_expr(cols: list[str]) -> pl.Expr:
    """
    Count mutated codons in a set of nt difference columns.
    A codon is mutated if any of its 3 positions is non-zero AND non-null.
    Gap characters (ASCII 45, '-') that differ between seq and germ are also counted;
    they are filtered out later during R/S classification (invalid codon translation).
    """
    assert len(cols) % 3 == 0, "Column list must be divisible into triplets"
    codon_flags = []
    for i in range(0, len(cols), 3):
        c1, c2, c3 = cols[i], cols[i+1], cols[i+2]
        codon_mutated = (
            (pl.col(c1).fill_null(0) != 0) |
            (pl.col(c2).fill_null(0) != 0) |
            (pl.col(c3).fill_null(0) != 0)
        ).cast(pl.UInt16)
        codon_flags.append(codon_mutated)
    return pl.sum_horizontal(codon_flags)


summary = mutations_sorted.select([
    pl.col(KEY),
    codon_mut_count_expr(H_CDR_COLS + H_FWR_COLS).alias('n_mut_H'),
    codon_mut_count_expr(L_CDR_COLS + L_FWR_COLS).alias('n_mut_L'),
    codon_mut_count_expr(H_CDR_COLS).alias('n_mut_CDR_H'),
    codon_mut_count_expr(H_FWR_COLS).alias('n_mut_FWR_H'),
    codon_mut_count_expr(L_CDR_COLS).alias('n_mut_CDR_L'),
    codon_mut_count_expr(L_FWR_COLS).alias('n_mut_FWR_L'),
])

summary = summary.with_columns(
    (pl.col('n_mut_H') + pl.col('n_mut_L')).alias('n_mut_total')
)

print(summary.describe())

shape: (9, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ seq_name  ┆ n_mut_H   ┆ n_mut_L   ┆ … ┆ n_mut_FWR ┆ n_mut_CDR ┆ n_mut_FWR ┆ n_mut_to │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ _H        ┆ _L        ┆ _L        ┆ tal      │
│ str       ┆ str       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 4839991   ┆ 4.839991e ┆ 4.839991e ┆ … ┆ 4.839991e ┆ 4.839991e ┆ 4.839991e ┆ 4.839991 │
│           ┆           ┆ 6         ┆ 6         ┆   ┆ 6         ┆ 6         ┆ 6         ┆ e6       │
│ null_coun ┆ 0         ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           ┆

### 0.6b  Replacement (R) vs Silent (S) classification

For each mutated codon: translate germline and sequence codons using the standard genetic code.
- **R**: amino acid changed  
- **S**: nucleotide changed but amino acid preserved  

Computed separately for CDR and framework regions (per chain).

In [22]:
# Standard genetic code: codon string → amino acid char
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

# Integer-keyed lookup: a*65536 + b*256 + c (ASCII) → AA ordinal
# Returns 0 for unknown/null codons
AA_LOOKUP = {0: 0}  # null sentinel
for codon, aa in GENETIC_CODE.items():
    k = ord(codon[0]) * 65536 + ord(codon[1]) * 256 + ord(codon[2])
    AA_LOOKUP[k] = ord(aa)
_aa_lookup_vec = np.vectorize(lambda k: AA_LOOKUP.get(int(k), 0), otypes=[np.int32])


def translate_codon_array(nt: np.ndarray) -> np.ndarray:
    """Translate (N, 3) uint8/int32 nt array → (N,) int32 AA ordinals. 0 = null/invalid."""
    keys = nt[:, 0].astype(np.int32) * 65536 + \
           nt[:, 1].astype(np.int32) * 256  + \
           nt[:, 2].astype(np.int32)
    return _aa_lookup_vec(keys)


def count_rs(germ_nt: np.ndarray, seq_nt: np.ndarray,
             codon_col_indices: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """
    Count replacement (R) and silent (S) codon mutations.

    Parameters
    ----------
    germ_nt, seq_nt : (N, L) int32 arrays, nucleotide ASCII values; 0 = null/gap
    codon_col_indices : 0-based column indices for the FIRST nt of each codon

    Returns
    -------
    n_R, n_S : (N,) int16 arrays
    """
    N = germ_nt.shape[0]
    n_R = np.zeros(N, dtype=np.int16)
    n_S = np.zeros(N, dtype=np.int16)

    for start in tqdm(codon_col_indices):
        g = germ_nt[:, start:start+3]   # (N, 3)
        s = seq_nt[:,  start:start+3]   # (N, 3)

        # Valid = both germline and sequence have non-null nt at all 3 positions
        valid = np.all(g > 0, axis=1) & np.all(s > 0, axis=1)
        has_mut = valid & np.any(g != s, axis=1)

        if not np.any(has_mut):
            continue

        g_aa = translate_codon_array(g)   # (N,)
        s_aa = translate_codon_array(s)   # (N,)
        valid_aa = (g_aa > 0) & (s_aa > 0)

        n_R += (has_mut & (g_aa != s_aa) & valid_aa).astype(np.int16)
        n_S += (has_mut & (g_aa == s_aa) & valid_aa).astype(np.int16)

    return n_R, n_S

print("Functions defined.")

Functions defined.


In [23]:
# Extract numpy arrays directly from aligned_joint — seq and germ columns come from the
# SAME joined DataFrame, so row counts are guaranteed identical. No filter+sort needed.
H_COLS_ALL = sorted(
    [c for c in pos_cols if c.startswith('H')], key=lambda c: int(c[1:])
)
L_COLS_ALL = sorted(
    [c for c in pos_cols if c.startswith('L')], key=lambda c: int(c[1:])
)

# Germline columns have the "_germ" suffix added by the join
GERM_H_COLS = [f"{c}_germ" for c in H_COLS_ALL]
GERM_L_COLS = [f"{c}_germ" for c in L_COLS_ALL]

# Filter to columns present in the joined table
joint_cols = set(aligned_joint.columns)
H_COLS_ALL   = [c for c in H_COLS_ALL   if c in joint_cols]
GERM_H_COLS  = [c for c in GERM_H_COLS  if c in joint_cols]
L_COLS_ALL   = [c for c in L_COLS_ALL   if c in joint_cols]
GERM_L_COLS  = [c for c in GERM_L_COLS  if c in joint_cols]

print("Extracting numpy arrays from joint alignment...")
seq_H  = aligned_joint.select(H_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)
germ_H = aligned_joint.select(GERM_H_COLS).fill_null(0).to_numpy().astype(np.int32)
seq_L  = aligned_joint.select(L_COLS_ALL).fill_null(0).to_numpy().astype(np.int32)
germ_L = aligned_joint.select(GERM_L_COLS).fill_null(0).to_numpy().astype(np.int32)

assert seq_H.shape == germ_H.shape, f"H shape mismatch: {seq_H.shape} vs {germ_H.shape}"
assert seq_L.shape == germ_L.shape, f"L shape mismatch: {seq_L.shape} vs {germ_L.shape}"
print(f"H arrays: seq={seq_H.shape}, germ={germ_H.shape}")
print(f"L arrays: seq={seq_L.shape}, germ={germ_L.shape}")

Extracting numpy arrays from joint alignment...
H arrays: seq=(4839991, 447), germ=(4839991, 447)
L arrays: seq=(4839991, 444), germ=(4839991, 444)


In [24]:
def region_codon_starts(region_cols: list[str], all_cols: list[str]) -> list[int]:
    """Return 0-based column indices for the first nt of each codon in region_cols."""
    col_to_idx = {c: i for i, c in enumerate(all_cols)}
    starts = []
    for i in range(0, len(region_cols), 3):
        c = region_cols[i]
        if c in col_to_idx:
            starts.append(col_to_idx[c])
    return starts


# Filter region column lists to those present in joint table
H_CDR = [c for c in H_CDR_COLS if c in set(H_COLS_ALL)]
H_FWR = [c for c in H_FWR_COLS if c in set(H_COLS_ALL)]
L_CDR = [c for c in L_CDR_COLS if c in set(L_COLS_ALL)]
L_FWR = [c for c in L_FWR_COLS if c in set(L_COLS_ALL)]

print("Computing R/S for H CDR...")
t0 = time.time()
R_CDR_H, S_CDR_H = count_rs(germ_H, seq_H, region_codon_starts(H_CDR, H_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for H FWR...")
t0 = time.time()
R_FWR_H, S_FWR_H = count_rs(germ_H, seq_H, region_codon_starts(H_FWR, H_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for L CDR...")
t0 = time.time()
R_CDR_L, S_CDR_L = count_rs(germ_L, seq_L, region_codon_starts(L_CDR, L_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

print("Computing R/S for L FWR...")
t0 = time.time()
R_FWR_L, S_FWR_L = count_rs(germ_L, seq_L, region_codon_starts(L_FWR, L_COLS_ALL))
print(f"  done in {time.time()-t0:.1f}s")

# Key comes from aligned_joint (same source as the numpy arrays)
rs_df = pl.DataFrame({
    KEY:          aligned_joint[KEY].to_list(),
    'n_R_CDR_H':  R_CDR_H,
    'n_S_CDR_H':  S_CDR_H,
    'n_R_FWR_H':  R_FWR_H,
    'n_S_FWR_H':  S_FWR_H,
    'n_R_CDR_L':  R_CDR_L,
    'n_S_CDR_L':  S_CDR_L,
    'n_R_FWR_L':  R_FWR_L,
    'n_S_FWR_L':  S_FWR_L,
})

rs_df = rs_df.with_columns([
    (pl.col('n_R_CDR_H') + pl.col('n_R_CDR_L')).alias('n_R_CDR'),
    (pl.col('n_S_CDR_H') + pl.col('n_S_CDR_L')).alias('n_S_CDR'),
    (pl.col('n_R_FWR_H') + pl.col('n_R_FWR_L')).alias('n_R_FWR'),
    (pl.col('n_S_FWR_H') + pl.col('n_S_FWR_L')).alias('n_S_FWR'),
])

print("\nR/S summary:")
print(rs_df.select(['n_R_CDR','n_S_CDR','n_R_FWR','n_S_FWR']).describe())

Computing R/S for H CDR...


  0%|          | 0/57 [00:00<?, ?it/s]

  done in 106.8s
Computing R/S for H FWR...


  0%|          | 0/92 [00:00<?, ?it/s]

  done in 187.5s
Computing R/S for L CDR...


  0%|          | 0/60 [00:00<?, ?it/s]

  done in 109.4s
Computing R/S for L FWR...


  0%|          | 0/88 [00:00<?, ?it/s]

  done in 181.2s

R/S summary:
shape: (9, 5)
┌────────────┬────────────┬────────────┬────────────┬────────────┐
│ statistic  ┆ n_R_CDR    ┆ n_S_CDR    ┆ n_R_FWR    ┆ n_S_FWR    │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ str        ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪════════════╪════════════╪════════════╡
│ count      ┆ 4.839991e6 ┆ 4.839991e6 ┆ 4.839991e6 ┆ 4.839991e6 │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ mean       ┆ 4.119767   ┆ 1.283866   ┆ 5.405461   ┆ 2.873414   │
│ std        ┆ 3.845253   ┆ 1.579354   ┆ 6.363777   ┆ 3.722127   │
│ min        ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ 25%        ┆ 1.0        ┆ 0.0        ┆ 0.0        ┆ 0.0        │
│ 50%        ┆ 3.0        ┆ 1.0        ┆ 3.0        ┆ 1.0        │
│ 75%        ┆ 7.0        ┆ 2.0        ┆ 9.0        ┆ 4.0        │
│ max        ┆ 30.0       ┆ 18.0       ┆ 90.0       ┆ 44.0       │
└────────────┴───

## 0.7  CDRH3 length

CDRH3 amino acid length = number of **occupied** nt positions in CDR3-H (Aho 109–137) ÷ 3.

iggnition encodes two distinct "absent" states:
- `null`: position is outside the alignment space entirely
- `45` (ASCII `'-'`): position exists in the Aho numbering but is a gap for this sequence

Both must be excluded when counting occupied positions, otherwise gap-padded CDR3 columns
would inflate length estimates.

In [ ]:
cdr3_H_cols = [c for c in H_REGIONS['CDR3'] if c in joint_cols]
cdr3_L_cols = [c for c in L_REGIONS['CDR3'] if c in joint_cols]

def cdr3_len_expr(cols: list[str], name: str) -> pl.Expr:
    """Count occupied (non-null, non-gap) nt positions in a CDR3 region, divide by 3."""
    return (
        pl.sum_horizontal([
            (pl.col(c).is_not_null() & (pl.col(c) != GAP)).cast(pl.UInt16)
            for c in cols
        ]) // 3
    ).alias(name)

cdrh3_len = (
    aligned_joint
    .select([KEY] + cdr3_H_cols + cdr3_L_cols)
    .with_columns([
        cdr3_len_expr(cdr3_H_cols, 'cdrh3_length'),
        cdr3_len_expr(cdr3_L_cols, 'cdrl3_length'),
    ])
    .select([KEY, 'cdrh3_length', 'cdrl3_length'])
)

print(cdrh3_len.select(['cdrh3_length', 'cdrl3_length']).describe())
print(f"\nCDRH3 = 0: {cdrh3_len.filter(pl.col('cdrh3_length') == 0).height:,}")
print(f"CDRL3 = 0: {cdrh3_len.filter(pl.col('cdrl3_length') == 0).height:,}")

## 0.8  Quality control

Remove sequences that would corrupt downstream statistics:
1. Stop codons in V region (detected as `*` in translation)
2. Missing heavy **or** light chain alignment (null in H1 or L1)
3. Ambiguous germline assignment (if confidence score available)

Ambiguous-nucleotide sequences (N content) should have been removed by iggnition during alignment;
we verify below.

In [ ]:
# 1. Stop codons in VH (FR1–FR3, Aho 1–108)
H_V_COLS = [c for c in H_REGIONS['FR1'] + H_REGIONS['CDR1'] + H_REGIONS['FR2'] +
                        H_REGIONS['CDR2'] + H_REGIONS['FR3'] if c in set(H_COLS_ALL)]

print("Detecting stop codons in VH region...")
STOP_AA = ord('*')
seq_H_V = seq_H[:, :len(H_V_COLS)]

has_stop = np.zeros(seq_H.shape[0], dtype=bool)
for start in range(0, len(H_V_COLS) - 2, 3):
    s_aa = translate_codon_array(seq_H_V[:, start:start+3])
    has_stop |= (s_aa == STOP_AA)

n_stop = has_stop.sum()
print(f"  Stop codons in VH:      {n_stop:,}")

stop_names = set(
    pl.Series(aligned_joint[KEY].to_list())
    .filter(pl.Series(has_stop))
    .to_list()
)

# 2. Missing chain alignment (null H1 or L1 = chain failed entirely)
missing_H = aligned_joint.filter(pl.col('H1').is_null()).select(KEY).to_series().to_list()
missing_L = aligned_joint.filter(pl.col('L1').is_null()).select(KEY).to_series().to_list()
print(f"  Missing H alignment:    {len(missing_H):,}")
print(f"  Missing L alignment:    {len(missing_L):,}")

# 3. CDRH3 or CDRL3 length = 0 (truncated / incomplete assembly)
zero_cdrh3 = cdrh3_len.filter(pl.col('cdrh3_length') == 0).select(KEY).to_series().to_list()
zero_cdrl3 = cdrh3_len.filter(pl.col('cdrl3_length') == 0).select(KEY).to_series().to_list()
print(f"  CDRH3 length = 0:       {len(zero_cdrh3):,}")
print(f"  CDRL3 length = 0:       {len(zero_cdrl3):,}")

fail_qc = stop_names | set(missing_H) | set(missing_L) | set(zero_cdrh3) | set(zero_cdrl3)
print(f"\nTotal failing QC: {len(fail_qc):,}  "
      f"(retaining {aligned_joint.height - len(fail_qc):,} sequences)")

## 0.9  Assemble master table

In [ ]:
meta = df.rename({'name': KEY})

master = (
    mutations_sorted
    .select(KEY)
    .join(meta,       on=KEY, how='left')
    .join(summary,    on=KEY, how='left')
    .join(rs_df,      on=KEY, how='left')
    .join(cdrh3_len,  on=KEY, how='left')   # adds cdrh3_length + cdrl3_length
    .filter(~pl.col(KEY).is_in(list(fail_qc)))
)

print(f"Master table: {master.shape}")
print(master.head(3))

## 0.10  Validation sanity checks

Expected patterns:
- Naïve sequences → mutation counts near zero
- Memory sequences → VH mutation peak at 5–15 codons
- CDR enrichment > FWR on average in memory (positive selection signal)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── 1. VH mutation load by naive/memory subset ──────────────────────────────
ax = axes[0]
subsets = {
    'naive_strict': master.filter(pl.col('naive_strict'))['n_mut_H'].to_numpy(),
    'naive_bio':    master.filter(pl.col('naive_bio') & ~pl.col('naive_strict'))['n_mut_H'].to_numpy(),
    'naive_comp':   master.filter(pl.col('naive_comp') & ~pl.col('naive_strict'))['n_mut_H'].to_numpy(),
    'memory':       master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp'))['n_mut_H'].to_numpy(),
}
colors = ['#2196F3', '#64B5F6', '#90CAF9', '#E53935']
for (label, vals), col in zip(subsets.items(), colors):
    ax.hist(vals, bins=range(0, 51), alpha=0.7, color=col, density=True,
            label=f"{label} (n={len(vals):,})", histtype='stepfilled')
ax.set_xlabel('VH codon mutations')
ax.set_ylabel('Density')
ax.set_title('VH mutation load by subset')
ax.legend(fontsize=7)
ax.set_xlim(0, 50)

# ── 2. CDR enrichment in memory sequences ───────────────────────────────────
ax = axes[1]
mem = master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp') & (pl.col('n_mut_H') > 0))
cdr_frac = (mem['n_mut_CDR_H'].cast(pl.Float32) / mem['n_mut_H'].cast(pl.Float32)).drop_nulls().to_numpy()
ax.hist(cdr_frac, bins=50, color='steelblue', edgecolor='none')
# Neutral expectation = fraction of V-region positions in CDRs
neutral = len(H_CDR_COLS) / (len(H_CDR_COLS) + len(H_FWR_COLS))
ax.axvline(neutral, color='red', linestyle='--', linewidth=1.5,
           label=f'Neutral ({neutral:.2f})')
ax.set_xlabel('CDR fraction of VH mutations')
ax.set_ylabel('Count (memory sequences)')
ax.set_title('CDR mutation enrichment (memory)')
ax.legend()

# ── 3. CDRH3 and CDRL3 length distributions ─────────────────────────────────
ax = axes[2]
h3 = master.filter(pl.col('cdrh3_length') > 0)['cdrh3_length'].to_numpy()
l3 = master.filter(pl.col('cdrl3_length') > 0)['cdrl3_length'].to_numpy()
bins = range(0, 35)
ax.hist(h3, bins=bins, alpha=0.6, color='#E53935', label=f'CDRH3 (n={len(h3):,})', density=True)
ax.hist(l3, bins=bins, alpha=0.6, color='#1E88E5', label=f'CDRL3 (n={len(l3):,})', density=True)
ax.set_xlabel('CDR3 length (aa)')
ax.set_ylabel('Density')
ax.set_title('CDR3 length distributions')
ax.legend()

plt.tight_layout()
plt.savefig(PROC_DIR / 'step0_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## 0.11  Save

In [29]:
master.write_parquet(MASTER_FILE)
print(f"Saved master table → {MASTER_FILE}")
print(f"Shape: {master.shape}")
print(f"\nColumn list:")
for c in master.columns:
    print(f"  {c}")

Saved master table → /home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/processed/aligned_master.parquet
Shape: (4839942, 330)

Column list:
  seq_name
  sequence_id:0
  v_gene:0
  d_gene:0
  j_gene:0
  c_gene:0
  cdr3_length:0
  junction_aa:0
  v_identity:0
  v_identity_aa:0
  d_identity:0
  d_identity_aa:0
  j_identity:0
  j_identity_aa:0
  productive:0
  complete_vdj:0
  sequence:0
  germline:0
  sequence_vdjc:0
  germline_vdjc:0
  sequence_lvdjc:0
  germline_lvdjc:0
  sequence_aa:0
  germline_aa:0
  sequence_vdjc_aa:0
  germline_vdjc_aa:0
  sequence_lvdjc_aa:0
  germline_lvdjc_aa:0
  sequence_gapped:0
  germline_gapped:0
  sequence_vdjc_gapped:0
  germline_vdjc_gapped:0
  sequence_lvdjc_gapped:0
  germline_lvdjc_gapped:0
  sequence_alignment:0
  germline_alignment:0
  sequence_alignment_aa:0
  germline_alignment_aa:0
  umi:0
  quality:0
  locus:0
  frame:0
  species:0
  germline_database:0
  sequence_input:0
  sequence_oriented:0
  rev_comp:0
  productivity_issues:0
  stop_co